<a href="https://colab.research.google.com/github/Kaveesha-Vihanga/DS_Project/blob/component-3/component_3_data_cleaning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!pip install langdetect

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 7.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for langdetect: filename=langdetect-1.0.9-py3-none-any.whl size=993223 sha256=56737db829302c4c3b3631acc917a0f02f8b81d0358eef519c9885050ad36bf7
  Stored in directory: /root/.cache/pip/wheels/c1/67/88/e844b5b022812e15a52e4eaa38a1e709e99f06f6639d7e3ba7
Successfully built langdetect


In [ ]:
# ============================================
# DIAGNOSTIC PREPROCESSING SCRIPT
# ============================================

import pandas as pd
import numpy as np
import re

# Load data
df = pd.read_csv('/content/drive/MyDrive/DSGP component 3/Dataset.csv')
print(f"Original shape: {df.shape}")

# ============================================
# Step 1: Parse dates (with diagnostics)
# ============================================
def parse_review_date(text):
    if pd.isna(text):
        return pd.NaT
    # Remove "Reviewed: " and parse
    cleaned = re.sub(r'^Reviewed:\s*', '', str(text)).strip()
    try:
        return pd.to_datetime(cleaned, errors='coerce')
    except:
        return pd.NaT

def parse_stay_date(text):
    if pd.isna(text):
        return pd.NaT
    # Extract month year after bullet
    match = re.search(r'·\s*(\w+\s+\d{4})', str(text))
    if match:
        date_str = match.group(1)
        try:
            return pd.to_datetime(date_str, format='%B %Y', errors='coerce')
        except:
            return pd.NaT
    return pd.NaT

# Parse both columns
df['review_date_parsed'] = df['Review Date'].apply(parse_review_date)
df['stay_date_parsed'] = df['Stay Date'].apply(parse_stay_date)

# Check how many parsed successfully
print(f"Review Date parsed: {df['review_date_parsed'].notna().sum()} valid")
print(f"Stay Date parsed: {df['stay_date_parsed'].notna().sum()} valid")

# Show sample parsed dates
print("\nSample parsed review dates:")
print(df[['Review Date', 'review_date_parsed']].dropna().head(10))

print("\nSample parsed stay dates:")
print(df[['Stay Date', 'stay_date_parsed']].dropna().head(10))

# Combine, preferring review date
df['clean_date'] = df['review_date_parsed'].fillna(df['stay_date_parsed'])
print(f"\nCombined valid dates: {df['clean_date'].notna().sum()}")

# ============================================
# Step 2: Check year distribution before filtering
# ============================================
if df['clean_date'].notna().any():
    years = df['clean_date'].dt.year
    print("\nYear distribution before filtering:")
    print(years.value_counts().sort_index())
else:
    print("No valid dates found. Stopping.")
    exit()

# ============================================
# Step 3: Filter by year 2015-2025
# ============================================
mask = (df['clean_date'].dt.year >= 2015) & (df['clean_date'].dt.year <= 2025)
df_filtered = df[mask].copy()
print(f"\nRows after year filter: {len(df_filtered)}")

if len(df_filtered) == 0:
    print("All rows were filtered out. Consider expanding year range if needed.")
    # You could optionally keep all years for now and adjust later.
    # df_filtered = df.copy()  # Uncomment to keep all years for testing
else:
    df = df_filtered

# Continue with rest of preprocessing only if we have data...
print(f"Rows after year filter: {len(df)}")
print(f"Year range: {df['clean_date'].dt.year.min()} – {df['clean_date'].dt.year.max()}")

# ============================================
# 4. COMBINE REVIEW FIELDS
# ============================================
print("\n" + "="*50)
print("COMBINING REVIEW FIELDS")
print("="*50)

review_title = df['Review Title'].fillna('').astype(str) if 'Review Title' in df.columns else ''
happy = df['Happy Review'].fillna('').astype(str) if 'Happy Review' in df.columns else ''
unhappy = df['Unhappy Review'].fillna('').astype(str) if 'Unhappy Review' in df.columns else ''

df['review_full'] = review_title + ' ' + happy + ' ' + unhappy
df['review_full'] = df['review_full'].str.replace(r'\s+', ' ', regex=True).str.strip()

# ============================================
# 5. HANDLE MISSING VALUES
# ============================================
print("\n" + "="*50)
print("HANDLING MISSING VALUES")
print("="*50)

# Drop rows where review text is empty
empty_review_mask = (df['review_full'] == '') | (df['review_full'].isna())
print(f"Rows with empty combined review: {empty_review_mask.sum()} (will be dropped)")
df = df[~empty_review_mask].copy()

# Fill categorical columns
categorical_fill_cols = ['Country', 'Traveller Type', 'Room Info', 'Hotel Name', 'Address']
for col in categorical_fill_cols:
    if col in df.columns:
        df[col] = df[col].fillna('Unknown')

# Handle Rating
if 'Rating' in df.columns:
    # Extract numeric rating if stored as string with symbols
    df['Rating'] = df['Rating'].astype(str).str.extract(r'(\d+\.?\d*)')[0].astype(float)

print(f"Missing values after handling:")
print(df.isnull().sum()[df.isnull().sum() > 0])

# ============================================
# 6. REMOVE DUPLICATES
# ============================================
print("\n" + "="*50)
print("REMOVING DUPLICATE REVIEWS")
print("="*50)

dup_cols = []
if 'Hotel Name' in df.columns:
    dup_cols.append('Hotel Name')
if 'User Name' in df.columns:
    dup_cols.append('User Name')
dup_cols.append('review_full')

df['review_lower'] = df['review_full'].str.lower()
duplicate_mask = df.duplicated(subset=dup_cols + ['review_lower'], keep='first')
print(f"Duplicate rows found: {duplicate_mask.sum()}")
df = df[~duplicate_mask].copy()
df.drop('review_lower', axis=1, inplace=True)

# ============================================
# 7. HANDLE OUTLIERS IN RATING
# ============================================
print("\n" + "="*50)
print("HANDLING OUTLIERS IN RATING")
print("="*50)

if 'Rating' in df.columns:
    print("Rating summary before:")
    print(df['Rating'].describe())

    # Detect scale (1-5 or 1-10)
    upper_bound = 10 if df['Rating'].quantile(0.95) > 5 else 5
    lower_bound = 1

    # Cap outliers
    df['Rating'] = df['Rating'].clip(lower=lower_bound, upper=upper_bound)

    # Impute missing with median
    median_rating = df['Rating'].median()
    df['Rating'] = df['Rating'].fillna(median_rating)

    print(f"Rating capped to [{lower_bound}, {upper_bound}] and missing filled with median {median_rating:.2f}")
    print("Rating summary after:")
    print(df['Rating'].describe())
else:
    print("No Rating column found.")

# ============================================
# 8. BASIC TEXT CLEANING (FOR XLM-RoBERTA)
# ============================================
print("\n" + "="*50)
print("BASIC TEXT CLEANING")
print("="*50)

def clean_text_for_xlm(text):
    if pd.isna(text):
        return ''
    text = re.sub(r'http\S+', '', text)
    text = re.sub(r'www\.\S+', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['review_clean'] = df['review_full'].apply(clean_text_for_xlm)

# ============================================
# 9. OPTIONAL LANGUAGE DETECTION (SKIP IF SLOW)
# ============================================
print("\n" + "="*50)
print("LANGUAGE DETECTION (FOR INFORMATION, NOT FILTERING)")
print("="*50)

def detect_language(text):
    """Return language code or 'unknown' if detection fails."""
    if pd.isna(text) or text.strip() == '':
        return 'unknown'
    try:
        return detect(text)
    except LangDetectException:
        return 'unknown'

# ============================================
# 10. SELECT FINAL COLUMNS
# ============================================
print("\n" + "="*50)
print("SELECTING FINAL COLUMNS")
print("="*50)

keep_cols = [
    'User Name', 'Country', 'Room Info', 'Stay Date', 'Traveller Type',
    'Rating', 'Review Date', 'Hotel Name', 'Address',
    'review_full', 'review_clean', 'language', 'clean_date'
]
keep_cols = [col for col in keep_cols if col in df.columns]

df_final = df[keep_cols].copy()
print(f"Final shape: {df_final.shape}")

# ============================================
# 11. SAVE CLEANED DATASET
# ============================================
output_filename = '/content/drive/MyDrive/DSGP component 3/sri_lanka_hotel_reviews_cleaned.csv'
df_final.to_csv(output_filename, index=False)
print(f"\n✅ Saved to {output_filename}")

# ============================================
# 12. SUMMARY
# ============================================
print("\n" + "="*50)
print("SUMMARY")
print("="*50)
print(f"Total reviews: {len(df_final)}")
print(f"Date range: {df_final['clean_date'].min().date()} to {df_final['clean_date'].max().date()}")
print(f"Unique hotels: {df_final['Hotel Name'].nunique() if 'Hotel Name' in df_final.columns else 'N/A'}")
print(f"Average rating: {df_final['Rating'].mean():.2f}" if 'Rating' in df.columns else "")

Original shape: (39646, 14)
Review Date parsed: 39646 valid
Stay Date parsed: 39646 valid

Sample parsed review dates:
                   Review Date review_date_parsed
0   Reviewed: January 20, 2026         2026-01-20
1  Reviewed: February 26, 2026         2026-02-26
2  Reviewed: February 23, 2026         2026-02-23
3  Reviewed: February 21, 2026         2026-02-21
4  Reviewed: February 21, 2026         2026-02-21
5  Reviewed: February 20, 2026         2026-02-20
6  Reviewed: February 20, 2026         2026-02-20
7   Reviewed: February 8, 2026         2026-02-08
8   Reviewed: January 30, 2026         2026-01-30
9   Reviewed: January 25, 2026         2026-01-25

Sample parsed stay dates:
                      Stay Date stay_date_parsed
0    1 night · \n\nJanuary 2026       2026-01-01
1  2 nights · \n\nFebruary 2026       2026-02-01
2   1 night · \n\nFebruary 2026       2026-02-01
3   1 night · \n\nFebruary 2026       2026-02-01
4   1 night · \n\nFebruary 2026       2026-02-01
5   1 nigh

In [ ]:
import pandas as pd
import re

# Load your cleaned dataset
df = pd.read_csv('/content/drive/MyDrive/DSGP component 3/sri_lanka_hotel_reviews_cleaned.csv')  # replace with your file name
print(f"Original shape: {df.shape}")

# 1. Remove rows with placeholder text
placeholder = "There are no comments available for this review"
mask_placeholder = df['review_clean'].str.contains(placeholder, case=False, na=False)
df = df[~mask_placeholder].copy()
print(f"After removing placeholder text: {len(df)}")

# 2. Calculate word count
df['word_count'] = df['review_clean'].str.split().str.len()

# 3. Keep only reviews with at least 5 words
min_words = 5
df_filtered = df[df['word_count'] >= min_words].copy()
print(f"After keeping ≥{min_words} words: {len(df_filtered)}")

# 4. (Optional) Check distribution
print("\nWord count stats after filtering:")
print(df_filtered['word_count'].describe())

# 5. Save filtered dataset
output_file = '/content/drive/MyDrive/DSGP component 3/sri_lanka_hotel_reviews_filtered.csv'
df_filtered.to_csv(output_file, index=False)
print(f"\n✅ Filtered dataset saved to: {output_file}")

Original shape: (32275, 11)
After removing placeholder text: 23199
After keeping ≥5 words: 22055

Word count stats after filtering:
count    22055.000000
mean        45.249875
std         45.375541
min          5.000000
25%         17.000000
50%         32.000000
75%         57.000000
max        617.000000
Name: word_count, dtype: float64

✅ Filtered dataset saved to: /content/drive/MyDrive/DSGP component 3/sri_lanka_hotel_reviews_filtered.csv


In [ ]:
import pandas as pd
from langdetect import detect, DetectorFactory
from langdetect.lang_detect_exception import LangDetectException
from tqdm import tqdm

# Set seed for reproducibility
DetectorFactory.seed = 42

# Load your filtered dataset (if not already loaded)
df = pd.read_csv('/content/drive/MyDrive/DSGP component 3/sri_lanka_hotel_reviews_filtered.csv')
print(f"Loaded {len(df)} reviews")

# Function to detect language
def detect_language(text):
    if pd.isna(text) or text.strip() == '':
        return 'unknown'
    try:
        return detect(text)
    except LangDetectException:
        return 'error'

# Apply language detection (show progress bar)
tqdm.pandas(desc="Detecting language")
df['language'] = df['review_clean'].progress_apply(detect_language)

# Show language distribution
print("\nLanguage distribution:")
print(df['language'].value_counts().head(10))

# Keep only English reviews
df_en = df[df['language'] == 'en'].copy()
print(f"\nEnglish reviews: {len(df_en)} ({len(df_en)/len(df)*100:.1f}%)")

# Save English-only dataset
output_file = '/content/drive/MyDrive/DSGP component 3/sri_lanka_hotel_reviews_english.csv'
df_en.to_csv(output_file, index=False)
print(f"✅ Saved English reviews to: {output_file}")

Loaded 22055 reviews


Detecting language: 100%|██████████| 22055/22055 [01:23<00:00, 264.75it/s]



Language distribution:
language
en    16252
de     1501
fr     1034
ru      788
es      479
it      474
nl      354
pl      211
ar      121
sv       84
Name: count, dtype: int64

English reviews: 16252 (73.7%)
✅ Saved English reviews to: /content/drive/MyDrive/DSGP component 3/sri_lanka_hotel_reviews_english.csv
